# 01 · 데이터 준비

통합 CSV → 위치 확인 → 행정구역 결합 → 행정동 집계 → 관측월별 주민등록인구.

분석 입력은 `데이터/전처리 데이터_통합본` 한 곳입니다. 제공 파일은 읽기 전용입니다. 좌표계 EPSG:5179는 공급자 확인 전의 임시 가정입니다.

패키지 설치는 README에 따라 한 번만 진행하세요. 커널을 재시작하고 위에서 아래로 실행합니다.

## 1. 경로와 기준 입력

**목적:** 공통 설정을 확인합니다. **입력:** 통합 CSV 3개와 카드 CSV 1개. **처리:** 파일 역할과 준비 상태를 확인합니다. **읽는 법:** 카드는 목록에만 포함되며 이번 통신 분석에는 사용하지 않습니다.

In [ ]:
from pathlib import Path
import sys

# 개인 PC 설정은 이 셀의 PROJECT_ROOT 한 곳에서만 바꿉니다.
# None이면 현재 폴더와 상위 폴더에서 프로젝트를 찾습니다.
PROJECT_ROOT = None
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                         if (p / "Code" / "sk_admin_mapping" / "project_settings.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("PROJECT_ROOT에 이 프로젝트의 루트 폴더를 지정하세요.")
PROJECT_ROOT = Path(PROJECT_ROOT).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "Code" / "sk_admin_mapping"))

from data_preparation import DataPreparation
from column_labels import show_table
preparation = DataPreparation(PROJECT_ROOT)
print("기준 입력:", preparation.input_directory)
print("기준 결과:", preparation.result_directory)


## 2. 관측 위치와 공간 매핑

**목적·처리:** 통합본의 모든 정규화 위치 키를 기존 인덱스와 대조합니다. 일치할 때만 공간 매핑을 재사용하며, 불일치하면 새로 생성합니다. 좌표 가정이 확인된 것으로 승격되지는 않습니다.

**입력:** 앞 단계의 결과. **결과 읽는 법:** 표의 단위·상태와 제외 사유를 함께 확인합니다.

In [ ]:
preparation.map_sources()


## 3. 매칭 품질

**목적·처리:** 월·지역·상태별 위치 수와 비율을 확인합니다. 미매칭/보류는 유동 0과 다릅니다.

**입력:** 앞 단계의 결과. **결과 읽는 법:** 표의 단위·상태와 제외 사유를 함께 확인합니다.

In [ ]:
preparation.summarize_quality()


## 4. 행정구역 결합

**목적·처리:** 통합 CSV의 행 수와 모든 원래 문자열을 보존합니다. 결합 전후 인구 값의 합계와 결측/비수치 개수를 대조합니다.

**입력:** 앞 단계의 결과. **결과 읽는 법:** 표의 단위·상태와 제외 사유를 함께 확인합니다.

In [ ]:
preparation.attach_sources()


## 5. 행정동 집계

**목적·처리:** 정상 매칭 위치만 집계합니다. 같은 테이블의 중복 위치는 전부 보류하며, 원본 행을 삭제하지 않습니다. 일평균 값의 위치별 합은 지역의 고유 인원수가 아닙니다.

**입력:** 앞 단계의 결과. **결과 읽는 법:** 표의 단위·상태와 제외 사유를 함께 확인합니다.

In [ ]:
preparation.aggregate_regions()


## 6. 관측월별 주민등록인구

**목적·처리:** 행안부 공식 조회의 2025년 7~12월 월말 총인구와 성·연령별 주민 수를 행정동·관측월로 결합합니다. 최초 실행은 공식 사이트 접속이 필요하며 이후 검증된 인구 캐시를 사용합니다. 외국인은 제외됩니다.

**입력:** 앞 단계의 결과. **결과 읽는 법:** 표의 단위·상태와 제외 사유를 함께 확인합니다.

In [ ]:
preparation.join_public_population()


## 선택 실행 · 검토용 지도

필요할 때 한 번만 표시합니다. 정규화 위치 표본과 행정구역의 관계를 확인하는 지도입니다.

In [ ]:
SHOW_REVIEW_MAP = True
if SHOW_REVIEW_MAP:
    preparation.review_map()
